# IntelliDoc AI — Notebook 1: FUNSD Dataset Validation

**Purpose:** Load the FUNSD dataset, check it is correctly structured, print statistics, and visualize sample annotated documents.

**Run this notebook first, before any model training.**

---
### Before Running This Notebook
1. Download FUNSD from: https://guillaumejaume.github.io/FUNSD/dataset.zip
2. Unzip it. You get a folder called `dataset/` containing `training_data/` and `testing_data/`
3. Upload these two folders to your **Google Drive** inside a folder called `funsd/`
4. Your Google Drive should look like:
```
MyDrive/
    funsd/
        training_data/
            images/
            annotations/
        testing_data/
            images/
            annotations/
```

## Step 1: Mount Google Drive
This connects your Google Drive to this Colab session so we can read your files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

## Step 2: Set Dataset Path and Install Libraries
Tell Python where your FUNSD folder is.

In [ ]:
import os

# Path to your FUNSD folder on Google Drive
FUNSD_ROOT = '/content/drive/MyDrive/funsd'
TRAIN_DIR  = os.path.join(FUNSD_ROOT, 'training_data')
TEST_DIR   = os.path.join(FUNSD_ROOT, 'testing_data')

# Quick sanity check
for path in [FUNSD_ROOT, TRAIN_DIR, TEST_DIR]:
    exists = os.path.exists(path)
    print(f'  {"EXISTS" if exists else "MISSING"} → {path}')

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError('FUNSD training_data folder not found! Please re-check your Google Drive path.')

In [ ]:
# Install required libraries
!pip install -q Pillow matplotlib
print('Libraries ready!')

## Step 3: Load All Annotations
FUNSD stores each document's labels as a JSON file. We will load all of them and inspect their structure.

In [ ]:
import json
import glob

def load_funsd_split(split_dir):
    """Load all annotation JSON files from a FUNSD split folder."""
    annotation_dir = os.path.join(split_dir, 'annotations')
    annotation_files = sorted(glob.glob(os.path.join(annotation_dir, '*.json')))
    documents = []
    for filepath in annotation_files:
        with open(filepath, 'r') as f:
            data = json.load(f)
            data['filename'] = os.path.basename(filepath).replace('.json', '')
            documents.append(data)
    return documents

train_docs = load_funsd_split(TRAIN_DIR)
test_docs  = load_funsd_split(TEST_DIR)

print(f'Training documents loaded : {len(train_docs)}')
print(f'Test documents loaded     : {len(test_docs)}')

## Step 4: Print Dataset Statistics
Count how many words exist, what label types are present, and how they are distributed.

In [ ]:
from collections import Counter

def compute_stats(documents, split_name):
    label_counter = Counter()
    total_words   = 0
    total_blocks  = 0

    for doc in documents:
        for form_item in doc.get('form', []):
            label = form_item.get('label', 'unknown')
            words = form_item.get('words', [])
            label_counter[label] += len(words)
            total_words  += len(words)
            total_blocks += 1

    print(f'\n=== {split_name} Split Statistics ===')
    print(f'  Documents      : {len(documents)}')
    print(f'  Labeled blocks : {total_blocks}')
    print(f'  Total words    : {total_words}')
    print(f'  Label breakdown:')
    for label, count in label_counter.most_common():
        pct = 100 * count / total_words if total_words > 0 else 0
        print(f'    {label:<12} {count:>6} words  ({pct:.1f}%)')

compute_stats(train_docs, 'Training')
compute_stats(test_docs,  'Test')

## Step 5: Look at One Raw Annotation
Print the raw JSON for the first document so you can understand the data format.

In [ ]:
import pprint

sample_doc = train_docs[0]
print(f"Document filename : {sample_doc['filename']}")
print(f"Number of form items: {len(sample_doc.get('form', []))}")
print()
print('First 3 form items:')
pprint.pprint(sample_doc['form'][:3])

## Step 6: Visualize a Document with Bounding Boxes
Draw colored boxes on the document image so you can visually confirm the annotations are correct.

**Color legend:**
- 🟢 Green = `answer`
- 🔵 Blue = `question`
- 🟠 Orange = `header`
- ⚪ Gray = `other`

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

LABEL_COLORS = {
    'answer'  : (0, 200, 0, 120),     # Green
    'question': (0, 0, 200, 120),     # Blue
    'header'  : (255, 140, 0, 120),   # Orange
    'other'   : (150, 150, 150, 120), # Gray
}

def visualize_funsd_document(doc, split_dir, doc_index=0):
    filename = doc['filename']
    image_path = os.path.join(split_dir, 'images', filename + '.png')

    if not os.path.exists(image_path):
        print(f'Image not found: {image_path}')
        return

    img    = Image.open(image_path).convert('RGBA')
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw   = ImageDraw.Draw(overlay)

    for form_item in doc.get('form', []):
        label = form_item.get('label', 'other')
        color = LABEL_COLORS.get(label, (128, 128, 128, 100))
        box   = form_item.get('box', None)
        if box:
            draw.rectangle(box, fill=color, outline=color[:3] + (255,), width=2)

    combined = Image.alpha_composite(img, overlay).convert('RGB')

    plt.figure(figsize=(12, 16))
    plt.imshow(combined)
    plt.title(f'Document: {filename}  |  Green=answer  Blue=question  Orange=header  Gray=other',
              fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Displayed document: {filename}')

# Visualize the first training document
visualize_funsd_document(train_docs[0], TRAIN_DIR)

## Step 7: Visualize a Few More Documents
Change the index to see different documents from the training set.

In [ ]:
# Visualize documents at index 1, 2, and 3
for i in [1, 2, 3]:
    if i < len(train_docs):
        visualize_funsd_document(train_docs[i], TRAIN_DIR)

## ✅ Validation Complete

If you can see:
- Correct document counts (149 training, 50 test)
- Label statistics showing `question`, `answer`, `header`, `other`
- Colored bounding boxes drawn on the document images

Then your FUNSD dataset is **correctly placed and ready** for model training.

**Next step:** Open `02_layoutlmv3_baseline.ipynb` to run the pretrained LayoutLMv3 and record baseline accuracy scores.